# Structural model of M. genitalium — reproducing Maritan 2022 with viva-parsimony

_Investigation `structural-model` — coder reproduction notebook._

**Question.** Can the Maritan et al. 2022 3D whole-cell structural model of Mycoplasma
genitalium be reproduced with viva-parsimony (the parsimony cellPACK-style
engine) in place of CellPACK — packing the same molecular inventory at true
abundance into a single-membrane cell — and does the viva_mgen whole-cell-model
reproduction's own simulated proteome yield a comparable 3D cell?

A structural investigation that reproduces the first 3D whole-cell model of
M. genitalium (Maritan 2022) using viva-parsimony. The Maritan supplement
(S1 proteins, S2 genes) supplies the ingredient roster, per-species
structures (curated PDB where available, else AlphaFold by UniProt), and the
circular genome; the pbg_parsimony octree engine packs them at true abundance
into a single-membrane capsule with one supercoiled nucleoid. Two studies
contrast the published WC-MG abundances (faithful baseline) against the
viva_mgen reproduction's simulated proteome.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-Mgen/viva-Mgen').is_dir():
    REPO = Path('/home/runner/work/viva-Mgen/viva-Mgen')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_mgen.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: s01 — Maritan baseline: the faithful 3D M. genitalium cell (parsimony) (`s01-maritan-baseline`)

**Question.** Packing the Maritan 2022 ingredient roster (S1) + genome (S2) at the published
WC-MG copy numbers with the parsimony engine, do we recover a faithful 3D
M. genitalium cell — hundreds of species placed at true abundance, a single
supercoiled nucleoid with RNAP sites at real genomic loci, and membrane
proteins in the bilayer?

**Objective.** Run the mgen_structural composite with counts_source=maritan, pack the cell
with the real parsimony binary, and confirm the pack places the expected number
of species at their WC-MG abundances with a single supercoiled chromosome seeded
from the S2 genome.

**Hypothesis.** The Maritan roster resolved to PDB/AlphaFold structures and packed into a
single-membrane ~0.3 µm capsule reproduces Maritan's mesoscale cell: a crowded
cytoplasm of hundreds of species at true abundance and one supercoiled
chromosome, using viva-parsimony rather than CellPACK.

**Claim.** A faithful 3D M. genitalium cell (Maritan's 144.47 nm sphere) packs from the
Maritan roster with parsimony: 30,355 molecules, 506 species, 75 membrane
proteins, one supercoiled chromosome, and a protein volume occupancy of 0.144
— exactly Maritan Table 1's reported figure. All acceptance checks PASS.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.structural.mgen_structural` | 0 | counts_source=maritan |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.structural.mgen_structural`** — `spec_viva_mgen_composites_structural_mgen_structural` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.structural.mgen_structural` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s01-maritan-baseline ===
STUDY = 's01-maritan-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: s02 — Reproduction-driven: the 3D cell from viva_mgen's simulated proteome (`s02-reproduction-driven`)

**Question.** Driving the SAME Maritan roster + structures with abundances from a viva_mgen
simulated run (rather than the published WC-MG counts), does the reproduction's
own proteome yield a structurally comparable 3D cell — and where does it diverge?

**Objective.** Run the mgen_structural composite with counts_source=sim against a viva_mgen
steady-state run's protein_counts, pack the cell, and compare per-category
species counts and occupancy to the s01 baseline.

**Hypothesis.** The viva_mgen reproduction's simulated per-gene protein counts, packed through
the identical pipeline, produce a cell whose species roster and occupancy match
the Maritan baseline within the reproduction's known expression differences
(GTP-capped translation, Poisson decay), diverging mainly for the most
dynamically-regulated genes.

**Claim.** The reproduction's simulated proteome packs the SAME cell — 605/632 baseline
species present at comparable roster — but LESS crowded: protein occupancy
~0.04 vs the Maritan-faithful 0.144. The species inventory is structurally
self-consistent; the divergence is one of absolute abundance scale (the
reproduction's simulated per-gene counts are lower than the WC-MG published
numbers Maritan used), not of which molecules are present.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_mgen.composites.structural.mgen_structural` | 0 | counts_source=sim |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_mgen.composites.structural.mgen_structural`** — `spec_viva_mgen_composites_structural_mgen_structural` (a plain, editable dict)


_composite spec file for `viva_mgen.composites.structural.mgen_structural` not found under `viva_mgen/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: s02-reproduction-driven ===
STUDY = 's02-reproduction-driven'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")